# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We'll inspect the structured clinical, hormone, imaging, and genetic variant tables.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset name: {metadata['name']}\n\nDescription: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Record sets, fields, and columns are referenced by their `@id`.

Let's explore what record sets (tables) are available in this dataset and enumerate their fields.

In [ ]:
# List all record sets by their @id
record_sets = dataset.metadata.record_sets

print("Available record sets (@id and name):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")

# For each record set, list fields by their @id
for rs in record_sets:
    print(f"\nRecord set: {rs['name']} (@id: {rs['@id']})")
    fields = rs['fields']
    for field in fields:
        fname = field.get('name', field['@id'])
        print(f"  - field @id: {field['@id']}, name: {fname}, dataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities referenced by their `@id`, as described above.

Let's load each record set into a pandas DataFrame.

In [ ]:
# Gather list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded DataFrame for record set {record_set_id} (shape: {df.shape}):")
    print(df.columns.tolist())
    print(df.head(2))

# For demonstration below, choose the first record set as our main analysis table
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping. All columns referenced by their `@id`.

We'll:
- Select a numeric field (e.g., age at diagnosis)
- Filter for age > 20
- Normalize the field
- Group by a categorical attribute (e.g., presence/absence of 'infertility')

Find a numeric field and group field from the overview above.

In [ ]:
# Find field @id for age and for infertility from the fields listing
# Here, we assume the field @id for age is 'age_at_diagnosis', for infertility is 'infertility'. Adjust if different.
numeric_field_id = 'age_at_diagnosis'  # Example: use actual field @id
group_field_id = 'infertility'  # Example: use actual field @id

# Filter where age_at_diagnosis > 20
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > 20]
    print(f"Filtered records with {numeric_field_id} > 20:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns: {main_df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. 

For example, plot age at diagnosis histogram and visualize infertility split.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of age_at_diagnosis
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=8, kde=True)
    plt.title('Age at Diagnosis Distribution')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

# Compare age distribution by infertility
if numeric_field_id in main_df.columns and group_field_id in main_df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'Age at Diagnosis by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel('Age')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded dataset metadata and inspected the available record sets and their fields by `@id`.
- Extracted tables into pandas DataFrames, referencing each entity by their unique `@id`.
- Performed simple EDA: filtered and normalized a numeric field, grouped by a categorical field.
- Visualized age distribution and group differences.

This notebook demonstrates reproducible, FAIR exploration of a rare disease clinical dataset using Croissant schema and `mlcroissant`. For deeper analysis, expand by referencing more record sets or fields via their `@id` as described.